In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "models").exists() and (candidate / "modules").exists():
            return candidate
    raise RuntimeError("Could not find repository root")

ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.append(str(ROOT / "modules"))
ROOT

In [1]:
import re

ROOT = Path.cwd().resolve()
import sys
import json
import numpy as np
from cobra.io import validate_sbml_model
from cobra.io import read_sbml_model
from cobra.io import write_sbml_model


from cobra.flux_analysis import flux_variability_analysis

import os
sys.path.append(os.path.join(ROOT, "modules"))

from utils import correct_ids, model_trimming

In [2]:
eps_tolerance = 1e-9
bound_scaling = 10.0

# In the original model I manually fixed a GPR
# GENE_ASSOCIATION: Mbar_A0818 or Mbar_A1889 )
original_model_path = str(ROOT / "models" / "M_barkeri_model_iMG746" / "mba_model.xml")
output_model_path   = str(ROOT / "models" / "M_barkeri_model_iMG746.xml")
final_model_path   = str(ROOT / "models" / "M_barkeri_model_iMG746_trimmed.xml")

print(f"Reading and validating SBML model from {original_model_path}")
mb_model = read_sbml_model(output_model_path)
mb_model, errors = validate_sbml_model(original_model_path)

assert len(errors["SBML_FATAL"]) == 0
assert len(errors["SBML_ERROR"]) == 0
assert len(errors["SBML_SCHEMA_ERROR"]) == 0

replace_dict = {
    "_LPAREN_":"_", 
    "_RPAREN_":"", 
    "_DASH_":"_" , 
    "_unknown_":"_",
    "unknown_": ""
}

print("Correcting model IDs")
for pattern, fix in replace_dict.items():
    print(f"- Replacing \"{pattern}\" by \"{fix}\" in model's reaction/metabolite ids")
    mb_model = correct_ids(mb_model, pattern, fix)
    print(f"- ID correcting pattern {pattern}.")

print(f"Writing fixed model to {output_model_path}")
write_sbml_model(mb_model, output_model_path)

print(f"Re-reading fixed model from {output_model_path}")
mb_model = read_sbml_model(output_model_path)

print("Final check for inconsistent IDs")
total = 0
for pattern, fix in replace_dict.items():
    for r in mb_model.reactions:
        if re.search(r.id, pattern):
            total += 1
    for m in mb_model.metabolites:
        if re.search(m.id, pattern):
            total += 1

assert total == 0
print("==========================================")
print("Success!!!")

tolerance = 1e-9

Reading and validating SBML model from models/M_barkeri_model_iMG746/mba_model.xml


SBML warnings in validation, check error log for details.
COBRA warnings in validation, check error log for details.


Correcting model IDs
- Replacing "_LPAREN_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 78
N of corrected metabolites ids 82
- ID correcting pattern _LPAREN_.
- Replacing "_RPAREN_" by "" in model's reaction/metabolite ids
N of corrected reaction ids 78
N of corrected metabolites ids 82
- ID correcting pattern _RPAREN_.
- Replacing "_DASH_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 4
N of corrected metabolites ids 85
- ID correcting pattern _DASH_.
- Replacing "_unknown_" by "_" in model's reaction/metabolite ids
N of corrected reaction ids 2
N of corrected metabolites ids 2
- ID correcting pattern _unknown_.
- Replacing "unknown_" by "" in model's reaction/metabolite ids
N of corrected reaction ids 0
N of corrected metabolites ids 4
- ID correcting pattern unknown_.
Writing fixed model to models/M_barkeri_model_iMG746.xml
Re-reading fixed model from models/M_barkeri_model_iMG746.xml
Final check for inconsistent IDs
Success!!!


In [3]:
H2_uptake_bound = -20
acetate_uptake_bound = -3

mb_model = read_sbml_model(output_model_path)

mb_model.reactions.ATPM.lower_bound    = 0

mb_model.reactions.EX_h2_e.lower_bound   = H2_uptake_bound

mb_model.reactions.EX_4abz_e.lower_bound    = -1
mb_model.reactions.EX_co2_e.lower_bound     = -100
mb_model.reactions.EX_cobalt2_e.lower_bound = -1
mb_model.reactions.EX_h2s_e.lower_bound     = -10
mb_model.reactions.EX_nac_e.lower_bound     = -10
mb_model.reactions.EX_nh4_e.lower_bound     = -10
mb_model.reactions.EX_ni2_e.lower_bound     = -10
mb_model.reactions.EX_pi_e.lower_bound      = -10
mb_model.reactions.EX_meoh_e.lower_bound    = 0
mb_model.reactions.EX_cys_L_e.lower_bound   = 0



print("Fixing bounds to remove meoh from the media and cysteine and meoh enable H2\n")

mb_model.reactions.get_by_id("EX_ac_e").lower_bound = 0
solution_co2 = mb_model.optimize()
summary_co2 = mb_model.summary()
print(f"Growth rate with co2 as carbon source: {solution_co2.objective_value:.5f}")

mb_model.reactions.get_by_id("EX_ac_e").lower_bound = acetate_uptake_bound
solution_acetate = mb_model.optimize()
summary_acetate = mb_model.summary()
print(f"Growth rate with Acetate (uptake:{acetate_uptake_bound}) as carbon source: {solution_acetate.objective_value:.5f}")


Fixing bounds to remove meoh from the media and cysteine and meoh enable H2

Growth rate with co2 as carbon source: 0.02725
Growth rate with Acetate (uptake:-3) as carbon source: 0.04360


In [4]:
print(f"Trimming model {mb_model.id}")

trimmed_model = model_trimming(mb_model, bound_scaling = 10)
for r in trimmed_model.exchanges:
    if r.upper_bound > 0:
        r.upper_bound = 100

trimmed_model.reactions.Mb_biomass_65.upper_bound = 0.05

trimmed_model.reactions.EX_4abz_e.lower_bound    = -1
trimmed_model.reactions.EX_co2_e.lower_bound     = -100
trimmed_model.reactions.EX_cobalt2_e.lower_bound = -1
trimmed_model.reactions.EX_h2s_e.lower_bound     = -10
trimmed_model.reactions.EX_nac_e.lower_bound     = -10
trimmed_model.reactions.EX_nh4_e.lower_bound     = -10
trimmed_model.reactions.EX_ni2_e.lower_bound     = -10
trimmed_model.reactions.EX_pi_e.lower_bound      = -10

print("Testing trimmed model")
trimmed_model.reactions.get_by_id("EX_ac_e").lower_bound = 0
new_solution_co2 = trimmed_model.optimize()
print(f"Growth rate with CO2 as carbon source:")
print(f"- Original solution: {solution_co2.objective_value:.6f}")
print(f"- Constrained-model solution: {new_solution_co2.objective_value:.6f}")

diff = solution_co2.objective_value - new_solution_co2.objective_value
assert abs(diff) < 1e-5

trimmed_model.reactions.get_by_id("EX_ac_e").lower_bound = acetate_uptake_bound
new_solution_ac = trimmed_model.optimize()
ac_summary = trimmed_model.summary()
print(f"Growth rate with Acetate (uptake:{acetate_uptake_bound}) as carbon source:")
print(f"- Original solution: {solution_acetate.objective_value:.6f}")
print(f"- Constrained-model solution: {new_solution_ac.objective_value:.6f}")

diff = solution_acetate.objective_value - new_solution_ac.objective_value
assert abs(diff) < 1e-5

print("=============================================")
print("Success!")

print(f"- N reactions in original model: {len(mb_model.reactions)}")
print(f"- N reactions in trimmed model:  {len(trimmed_model.reactions)}")
print()
print(f"- N metabolites in original model: {len(mb_model.metabolites)}")
print(f"- N metabolites in trimmed model:  {len(trimmed_model.metabolites)}")
print()
print(f"- N genes in original model: {len(mb_model.genes)}")
print(f"- N genes in trimmed model:  {len(trimmed_model.genes)}")

print(f"Writing Final model to {final_model_path}")
for m in trimmed_model.metabolites:
    m.notes = {}
write_sbml_model(trimmed_model, final_model_path)
print("Success!")

Trimming model Mb_iUPDATE
Removing blocked 265 reactions
Removing 259 gap metabolites
Removing 204 unconnected genes
Setting bounds
Testing trimmed model
Growth rate with CO2 as carbon source:
- Original solution: 0.027247
- Constrained-model solution: 0.027249


/home/mponce/projects/devel/PhysiCelldFBA/venv/lib/python3.10/site-packages/cobra/core/group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


Growth rate with Acetate (uptake:-3) as carbon source:
- Original solution: 0.043595
- Constrained-model solution: 0.043597
Success!
- N reactions in original model: 816
- N reactions in trimmed model:  551

- N metabolites in original model: 718
- N metabolites in trimmed model:  459

- N genes in original model: 750
- N genes in trimmed model:  546
Writing Final model to models/M_barkeri_model_iMG746_trimmed.xml
Success!


In [5]:
trimmed_model.reactions.get_by_id("EX_ac_e").lower_bound = 0
new_solution_co2 = trimmed_model.optimize()
co2_summary = trimmed_model.summary()

trimmed_model.reactions.get_by_id("EX_ac_e").lower_bound = acetate_uptake_bound
new_solution_ac = trimmed_model.optimize()
ac_summary = trimmed_model.summary()